# 沖縄県用Jageocoder辞書 作成ツール（build_okinawa_jageocoder）

`jageocoder-converter` を使って、沖縄県（都道府県コード47）のみのJageocoderローカル辞書を作成し、
ZIPファイルとしてダウンロードするための補助Notebookです。

**このNotebookは`sheltermatch.ipynb`本体ではありません。**
`sheltermatch.ipynb` は住所→座標変換にJageocoderのローカル辞書を利用しますが、その辞書を作る処理は
ここに分離しています。作成した `okinawa_jageocoder.zip` は `sheltermatch.ipynb` でアップロードして使います。

## 使い方

1. 下の「利用者設定」で利用規約確認後 `ACCEPT_TERMS = True` にする
2. 「ランタイム → すべてのセルを実行」
3. `okinawa_jageocoder.zip` を保存する

Google Driveは使用しません。辞書はColabランタイム上の一時ディレクトリに作成するため、ランタイムが
終了すると消えます。必ずダウンロードしたZIPファイルを保存してください。

**重要: このNotebookは毎回実行するものではありません。** 辞書を新規作成する時、または更新したい時だけ
実行してください。データ量によっては辞書生成に数分〜数十分かかることがあります。

## 利用者設定

通常変更が必要な項目はこれだけです。

In [ ]:
# ===== 利用者設定 =====

# jageocoder-converterは辞書生成時に、元データの利用規約への同意を必要とします。
# 下記のドキュメントで利用規約の内容をご自身で確認したうえで、True に変更してください。
# 参考: https://github.com/t-sagara/jageocoder-converter
# False のままでは、辞書の生成は行われません。
ACCEPT_TERMS = False

print(f"ACCEPT_TERMS = {ACCEPT_TERMS}")

## 必要ライブラリのインストール

Google Colabに標準で入っていない `jageocoder` と `jageocoder-converter` をインストールします。

In [ ]:
%pip install -q jageocoder jageocoder-converter

## 辞書の生成

沖縄県（都道府県コード47）のみを対象にJageocoderローカル辞書を生成します。街区・地番レベル以上の
精度を確保するため `--no-gaiku` は指定せず、住居表示住所データも省略しません。

`ACCEPT_TERMS = True` の場合のみ、利用規約に同意済みとして生成を開始します（このセルを再実行した
場合は、生成先を安全に削除してから作り直します）。`ACCEPT_TERMS = False` のままの場合は、生成を
行わず案内を表示してこのセルを終えます（エラーにはなりません）。

In [ ]:
import os
import shutil

PREF_CODE = "47"
JAGEOCODER_DB_DIR = "/content/okinawa_db"

dictionary_generated = False

if not ACCEPT_TERMS:
    print("辞書生成はまだ行いません。")
    print("利用規約を確認後、Notebook先頭の「利用者設定」で")
    print("ACCEPT_TERMS = True")
    print("に変更して、再度実行してください。")
else:
    if os.path.isdir(JAGEOCODER_DB_DIR):
        shutil.rmtree(JAGEOCODER_DB_DIR)
    os.makedirs(JAGEOCODER_DB_DIR, exist_ok=True)

    print(f"沖縄県（都道府県コード {PREF_CODE}）のJageocoder辞書生成を開始します。")
    print("データ量によっては数分〜数十分かかることがあります。")
    print("ACCEPT_TERMS=True のため、利用規約に同意済みとして --quiet を付けて実行します。")

    !python -m jageocoder_converter convert --db-dir="{JAGEOCODER_DB_DIR}" --quiet {PREF_CODE}

    # IPython(Colab)の!実行は、コマンドが異常終了してもNotebookの処理は次に進んでしまうため、
    # 実行直後にIPythonが設定する終了コード(_exit_code)を明示的に確認する。
    # _exit_code が取得できない場合も含め、0以外・未定義は失敗として扱う（成功を誤判定しないため）。
    exit_code = globals().get("_exit_code", 1)
    if exit_code != 0:
        print(f"辞書生成コマンドが異常終了しました（終了コード: {exit_code}）。上記の出力を確認してください。")
    else:
        print("辞書生成コマンドが正常終了しました。")
        dictionary_generated = True

## 生成結果の確認

辞書が実際にJageocoderから読み込めるかを確認します。ここで確認できた場合のみ、次のセルでZIP化・
ダウンロードします。

In [ ]:
dictionary_ready = False

if not ACCEPT_TERMS:
    print("辞書がまだ生成されていないため、この処理をスキップしました。")
elif not dictionary_generated:
    print("辞書生成が完了していないため、確認をスキップします。上のセルの出力を確認してください。")
else:
    generated_files = os.listdir(JAGEOCODER_DB_DIR)
    if not generated_files:
        print("辞書ディレクトリが空です。生成に失敗している可能性があります。上のセルの出力を確認してください。")
    else:
        import jageocoder

        try:
            jageocoder.init(db_dir=JAGEOCODER_DB_DIR)
            print("Jageocoderで辞書を正常に読み込めることを確認しました。")
            dictionary_ready = True
        except Exception as error:
            print("辞書ファイルは生成されていますが、Jageocoderからの読み込みに失敗しました。")
            print(f"詳細: {error}")

## ZIP化とダウンロード

辞書が正常に読み込めた場合のみ、辞書ディレクトリをZIP化してブラウザへダウンロードします。ZIP内には
辞書ディレクトリの中身のみを格納し、余分な親ディレクトリは含めません（展開したディレクトリをそのまま
`jageocoder.init(db_dir=...)` に渡せる構造です）。

`ACCEPT_TERMS = False` のままの場合は、この処理をスキップします。

In [ ]:
ZIP_FILENAME = "okinawa_jageocoder.zip"

import zipfile

from google.colab import files

if not dictionary_ready:
    print("辞書がまだ利用できる状態になっていないため、ZIP化・ダウンロードをスキップしました。")
else:
    # 辞書ディレクトリの中身をそのままZIPのトップレベルに格納する（余分な親ディレクトリを含めない）。
    with zipfile.ZipFile(ZIP_FILENAME, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for root, _dirs, filenames in os.walk(JAGEOCODER_DB_DIR):
            for filename in filenames:
                file_path = os.path.join(root, filename)
                arcname = os.path.relpath(file_path, JAGEOCODER_DB_DIR)
                zip_file.write(file_path, arcname)

    print(f"辞書ディレクトリを '{ZIP_FILENAME}' にZIP化しました。")
    print("sheltermatch.ipynb でこのZIPファイルをアップロードして利用してください。")

    files.download(ZIP_FILENAME)